# DPO Fine-Tuned Model — Colab Inference Notebook

Self-contained notebook for testing the DPO-trained Qwen3-1.7B reasoning model on Google Colab. No repo code needed — only the three files below.

## What to upload to Colab

| File | What it is | How to get it |
|---|---|---|
| `inference.ipynb` | This notebook | already in repo |
| `dpo_model.pth` | Single-file checkpoint (state_dict + base model id) | run `python export_for_colab.py` locally |
| `sample_test.jsonl` | 8 test problems (5 GSM8K + 3 MATH-500) | produced by the same export script |

Place all three in the Colab working directory (drag-drop into the Files pane), or stash the `.pth` in Google Drive and edit the path in the **Config** cell below.

## Runtime recommendation

**Runtime → Change runtime type → T4 GPU (or better)**. Qwen3-1.7B in fp16 fits comfortably in T4's 16 GB VRAM. CPU also works but is very slow.

## How loading works

`dpo/train_dpo.py` saves the **full merged weights** (`merge_and_unload()` + `save_pretrained()`), so the architecture is identical to `Qwen/Qwen3-1.7B`. The notebook downloads the base from the HF Hub, then overwrites every parameter with the state_dict from your `.pth`. The tokenizer comes from the HF Hub as well — `dpo/train_dpo.py` saves it without modification so the Hub copy is equivalent.

## 1 · Install dependencies

In [ ]:
!pip install -q --upgrade transformers accelerate sentencepiece

## 2 · Config — paths to your uploaded files

Edit `CKPT_PATH` and `SAMPLE_DATA_PATH` if you stored the files somewhere other than the Colab working directory (e.g. Google Drive).

In [ ]:
import os

CKPT_PATH        = "dpo_model.pth"        # produced by export_for_colab.py
SAMPLE_DATA_PATH = "sample_test.jsonl"    # produced by export_for_colab.py
BASE_MODEL_NAME  = "Qwen/Qwen3-1.7B"      # architecture + tokenizer source (from HF Hub)

# Optional: mount Google Drive and load from there instead.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_PATH        = "/content/drive/MyDrive/dpo_model.pth"
    SAMPLE_DATA_PATH = "/content/drive/MyDrive/sample_test.jsonl"

assert os.path.exists(CKPT_PATH),        f"Missing checkpoint: {CKPT_PATH}"
assert os.path.exists(SAMPLE_DATA_PATH), f"Missing sample data: {SAMPLE_DATA_PATH}"
print(f"Checkpoint  : {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1024**3:.2f} GB)")
print(f"Sample data : {SAMPLE_DATA_PATH}")
print(f"Base model  : {BASE_MODEL_NAME}")

## 3 · Imports & device setup

T4 GPUs (Colab free tier) do **not** have efficient bf16 — we pick fp16 there. A100/L4/etc. with bf16 support get bf16. The original training was bf16, so dtype is cast appropriately when loading.

In [ ]:
import json, re, time, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    DEVICE_MAP = "auto"
else:
    DEVICE = torch.device("cpu")
    DTYPE  = torch.float32
    DEVICE_MAP = None

# Mirrors sft/eval_correct.DATASET_MAX_TOKENS exactly.
MAX_NEW_TOKENS = {"gsm8k": 2048, "math500": 8192, "external": 4096}

print(f"Device: {DEVICE} | dtype: {DTYPE} | CUDA bf16: {torch.cuda.is_available() and torch.cuda.is_bf16_supported()}")

## 4 · Answer-extraction helpers

Copied verbatim from the repo's `algo/equivalent_ans.py` so the notebook needs no other repo files. Local-only grading — no LLM judge — matching `sft/eval_correct.is_correct_local`.

In [ ]:
def _extract_boxed(text: str) -> str:
    """Last \\boxed{...}; handles nested braces like \\boxed{(3, \\frac{\\pi}{2})}."""
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1:
            break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                if depth == 0:
                    results.append(text[idx + 7:i])
                    start = i + 1
                    break
                depth -= 1
        else:
            break
    return results[-1].strip() if results else ""


def _normalize(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|\\degree|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\dfrac", r"\\frac", t)
    t = re.sub(r"\\tfrac", r"\\frac", t)
    t = re.sub(r"\$+", "", t)
    t = re.sub(r"\s+", "", t)
    return t.lower()


def is_correct_local(extracted: str, ground_truth: str) -> bool:
    if not extracted:
        return False
    if _normalize(extracted) == _normalize(ground_truth):
        return True
    try:
        return float(extracted.replace(",", "").strip()) == float(str(ground_truth).replace(",", "").strip())
    except (ValueError, TypeError):
        return False


def extract_answer(text: str, dataset: str) -> str:
    boxed = _extract_boxed(text)
    if boxed:
        return boxed
    if dataset == "gsm8k":
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m:
            return m.group(1).replace(",", "").strip()
    return ""


def strip_think(text: str) -> str:
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text

## 5 · Prompt builder

Exactly the prompt used during DPO training (`dpo/train_dpo.build_prompt`) and SFT eval (`sft/eval_correct.build_prompt`). Qwen3 needs `enable_thinking=True` so the generation prefix ends with `<think>\n`; the model then continues the reasoning chain and emits `</think>` + the boxed answer.

In [ ]:
def build_prompt(tokenizer, question: str) -> str:
    msgs = [{"role": "user", "content": question.strip()}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tokenizer.apply_chat_template(msgs, enable_thinking=True, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, **kwargs)

## 6 · Load the DPO model

Two-step:
1. Instantiate `Qwen/Qwen3-1.7B` from the HF Hub — gives us the correct architecture, config, and tokenizer (the DPO training did not modify any of these).
2. Load `dpo_model.pth` and `load_state_dict` it on top — this replaces the base weights with the DPO-trained weights.

Tokenizer settings match training/eval: `pad_token = eos_token`, `padding_side = "left"`.

In [ ]:
def load_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return tok


def load_base(name):
    print(f"Downloading base model from HF Hub: {name}")
    mdl = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype=DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    mdl.eval()
    return mdl


def load_dpo(name, ckpt_path):
    mdl = load_base(name)
    print(f"Loading DPO state_dict: {ckpt_path}")
    payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    # Cast each tensor to the runtime dtype (saved as bf16; T4 needs fp16, CPU needs fp32).
    state_dict = {k: v.to(DTYPE) for k, v in state_dict.items()}
    missing, unexpected = mdl.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  WARNING: {len(missing)} missing keys (first 3): {missing[:3]}")
    if unexpected:
        print(f"  WARNING: {len(unexpected)} unexpected keys (first 3): {unexpected[:3]}")
    if not missing and not unexpected:
        print("  All keys matched.")
    del state_dict, payload
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"  VRAM after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    return mdl


tokenizer = load_tokenizer(BASE_MODEL_NAME)
dpo_model = load_dpo(BASE_MODEL_NAME, CKPT_PATH)

## 7 · Load sample test data

JSONL with `{"question", "answer", "dataset"}` per line — same schema as `data/gsm8k/test.jsonl` and `data/MATH-500/test.jsonl` in the repo.

In [ ]:
with open(SAMPLE_DATA_PATH, encoding="utf-8") as f:
    SAMPLES = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(SAMPLES)} samples")
for i, s in enumerate(SAMPLES, 1):
    print(f"  [{i}] {s['dataset']:7s}  ans={s['answer']!r:30s}  q[:60]={s['question'][:60]!r}")

## 8 · Generation + display helpers

In [ ]:
@torch.no_grad()
def generate_one(model, tokenizer, question: str, dataset: str) -> dict:
    prompt = build_prompt(tokenizer, question)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    max_new = MAX_NEW_TOKENS.get(dataset, MAX_NEW_TOKENS["external"])
    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    in_len = enc["input_ids"].shape[1]
    new_toks = out[0][in_len:]
    response = tokenizer.decode(new_toks, skip_special_tokens=True)

    return {
        "prompt":      prompt,
        "response":    response,
        "extracted":   extract_answer(response, dataset),
        "answer_tail": strip_think(response),
        "new_tokens":  int(len(new_toks)),
        "seconds":     time.time() - t0,
    }


def show_result(idx, ex, result, model_label):
    sep = "=" * 78
    print(sep)
    print(f"#{idx:02d}  [{ex.get('dataset','external')}]  model: {model_label}")
    print(sep)
    print("QUESTION:")
    q = ex["question"]
    print(q[:600] + ("..." if len(q) > 600 else ""))
    print()
    expected = ex.get("answer", "")
    print(f"EXPECTED ANSWER : {expected!r}")
    print(f"MODEL EXTRACTED : {result['extracted']!r}")
    if expected:
        print(f"CORRECT         : {is_correct_local(result['extracted'], expected)}")
    print(f"TOKENS / TIME   : {result['new_tokens']} tok / {result['seconds']:.1f}s")
    print("-" * 78)
    print("MODEL OUTPUT (post-</think> tail):")
    print(result["answer_tail"][:400] or "(empty — </think> never closed)")
    print("-- full response excerpt --")
    print(result["response"][:800].rstrip() + ("..." if len(result["response"]) > 800 else ""))
    print()

## 9 · Run DPO model on sample test data

In [ ]:
dpo_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(dpo_model, tokenizer, ex["question"], ex["dataset"])
    dpo_results.append((ex, r))
    show_result(i, ex, r, "DPO")

dpo_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in dpo_results if e.get("answer"))
dpo_graded  = sum(1 for e, _ in dpo_results if e.get("answer"))
print(f"\nDPO accuracy: {dpo_correct}/{dpo_graded} = {dpo_correct/max(dpo_graded,1)*100:.1f}%")

## 10 · Base model vs DPO model — side-by-side

Free the DPO model, load the unmodified `Qwen/Qwen3-1.7B`, run the same questions. Same prompt, same decoding — the delta is purely the DPO training effect.

In [ ]:
# Free DPO model before loading base.
del dpo_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = load_base(BASE_MODEL_NAME)

base_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(base_model, tokenizer, ex["question"], ex["dataset"])
    base_results.append((ex, r))
    show_result(i, ex, r, "BASE")

base_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in base_results if e.get("answer"))
base_graded  = sum(1 for e, _ in base_results if e.get("answer"))
print(f"\nBASE accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%")

In [ ]:
# Comparison table.
row_fmt = "{:<3} {:<8} {:<22} {:<22} {:<22} {:<7} {:<7} {:<7} {:<7}"
print(row_fmt.format("#", "dataset", "expected", "base", "dpo", "base_ok", "dpo_ok", "b_tok", "d_tok"))
print("-" * 116)
tok_base = tok_dpo = 0
for i, ((eb, rb), (ed, rd)) in enumerate(zip(base_results, dpo_results), 1):
    bok = is_correct_local(rb["extracted"], eb["answer"]) if eb.get("answer") else False
    dok = is_correct_local(rd["extracted"], ed["answer"]) if ed.get("answer") else False
    tok_base += rb["new_tokens"]
    tok_dpo  += rd["new_tokens"]
    print(row_fmt.format(
        i, eb.get("dataset", ""),
        (eb.get("answer") or "")[:21],
        (rb["extracted"]   or "")[:21],
        (rd["extracted"]   or "")[:21],
        "Y" if bok else "N",
        "Y" if dok else "N",
        rb["new_tokens"],
        rd["new_tokens"],
    ))
n = len(SAMPLES)
print("-" * 116)
print(f"BASE accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%  avg tokens: {tok_base/n:.0f}")
print(f"DPO  accuracy: {dpo_correct}/{dpo_graded } = {dpo_correct /max(dpo_graded ,1)*100:.1f}%  avg tokens: {tok_dpo /n:.0f}")
print(f"DELTA       : {(dpo_correct-base_correct)/max(n,1)*100:+.1f} pp accuracy, {(tok_dpo-tok_base)/n:+.0f} tokens avg")

## 11 · External test data (CSV / JSONL)

If the teacher uploads their own test file to Colab, point this section at it. Accepted schemas:

* **JSONL** — one JSON object per line. Required: `question` (alias: `problem`, `prompt`). Optional: `answer` (alias: `ground_truth`, `answerKey`) and `dataset` (one of `gsm8k`, `math500`; default `external`).
* **CSV** — same column names. Without an `answer` column, results are shown without a correctness flag.

Aliases match the keys `eval_correct.py` already recognises (`r.get("question") or r.get("problem")`, `r.get("answer", r.get("answerKey", ""))`).

In [ ]:
QUESTION_KEYS = ("question", "problem", "prompt")
ANSWER_KEYS   = ("answer", "ground_truth", "answerKey")


def _pick(record, keys):
    for k in keys:
        if k in record and record[k] not in (None, ""):
            return record[k]
    return None


def load_external(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith(".jsonl"):
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
    elif path.lower().endswith(".csv"):
        import csv
        with open(path, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        raise ValueError(f"Unsupported extension: {path} (need .jsonl or .csv)")

    examples = []
    for r in rows:
        q = _pick(r, QUESTION_KEYS)
        if not q:
            continue
        examples.append({
            "question": str(q),
            "answer":   str(_pick(r, ANSWER_KEYS) or ""),
            "dataset":  str(r.get("dataset", "external")).lower(),
        })
    return examples


def run_external(path, model, tokenizer, model_label="DPO", limit=None):
    examples = load_external(path)
    if limit:
        examples = examples[:limit]
    print(f"Loaded {len(examples)} examples from {path}")
    out = []
    correct = n_graded = 0
    for i, ex in enumerate(examples, 1):
        r = generate_one(model, tokenizer, ex["question"], ex["dataset"])
        show_result(i, ex, r, model_label)
        out.append({**ex, **r})
        if ex["answer"]:
            n_graded += 1
            correct  += int(is_correct_local(r["extracted"], ex["answer"]))
    if n_graded:
        print(f"\nAccuracy: {correct}/{n_graded} = {correct/n_graded*100:.1f}%")
    else:
        print("\nNo ground-truth answers in file — accuracy not computed.")
    return out


# ── To use the DPO model on the external file, reload it (we freed it above). ──
# del base_model
# gc.collect(); torch.cuda.empty_cache()
# dpo_model = load_dpo(BASE_MODEL_NAME, CKPT_PATH)

# EXTERNAL_PATH = "teacher_test.jsonl"   # or "teacher_test.csv"
# external_outputs = run_external(EXTERNAL_PATH, dpo_model, tokenizer, model_label="DPO", limit=20)

### Optional — save external-run outputs to a file

In [ ]:
# OUTPUT_PATH = "inference_external_results.jsonl"
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     for row in external_outputs:
#         f.write(json.dumps({
#             "question":         row["question"],
#             "ground_truth":     row["answer"],
#             "extracted_answer": row["extracted"],
#             "model_answer":     row["response"],
#             "new_tokens":       row["new_tokens"],
#             "seconds":          row["seconds"],
#             "dataset":          row["dataset"],
#         }, ensure_ascii=False) + "\n")
# from google.colab import files
# files.download(OUTPUT_PATH)